In [ ]:
import xarray as xr
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import colors as pltcolors
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse, Rectangle
from scipy.stats import linregress, chi2, ttest_ind
from cartopy import crs as ccrs, feature as cfeat
import cmweather
from datetime import datetime as dt
from metpy.plots import USCOUNTIES

seabreeze_side_select = 'all' # 'all', 'continental', 'maritime', 'crossing'
should_weight = True # for bivariate histogram

### Add more variables

In [ ]:
all_tracks = xr.open_dataset('/Volumes/LtgSSD/tobac_saves/all_tracks.zarr', engine='zarr')

all_tracks['track_dist_from_radar'] = (all_tracks['track_projection_x_coordinate']**2 + all_tracks['track_projection_y_coordinate']**2)**0.5
if seabreeze_side_select == 'all':
    seabreeze_mask = np.ones(all_tracks['track'].size, dtype=bool)
elif seabreeze_side_select == 'continental':
    seabreeze_mask = all_tracks.track_seabreeze.max(dim='timestep', skipna=True) == -2
elif seabreeze_side_select == 'maritime':
    seabreeze_mask = all_tracks.track_seabreeze.min(dim='timestep', skipna=True) == -1
elif seabreeze_side_select == 'crossing':
    seabreeze_mask = (all_tracks.track_seabreeze.min(dim='timestep', skipna=True) != -1) & (all_tracks.track_seabreeze.max(dim='timestep', skipna=True) != -2)
else:
    raise ValueError(f"Invalid seabreeze_side_select value: {seabreeze_side_select}")
all_tracks = all_tracks.isel(track=seabreeze_mask)
close_to_khgx_mask = (all_tracks['track_dist_from_radar'].min(dim='timestep', skipna=True)/1000 <= 90)
all_tracks = all_tracks.isel(track=close_to_khgx_mask)
all_tracks['track_area'] = all_tracks['track_area'] * (0.25)**2  # convert to km^2
# isolated_convective_mask = (all_tracks['track_area'].max(dim='timestep') < 450)
# all_tracks = all_tracks.isel(track=isolated_convective_mask)

all_tracks['track_day'] = all_tracks.time.min(dim='timestep', skipna=True).astype('datetime64[D]')
all_tracks['track_echotop'] = all_tracks['track_echotop'] / 1000  # convert to km
all_tracks['time_since_midnight'] = (all_tracks['time'] - all_tracks['time'].data.astype('datetime64[D]')).astype(float)/1e9/3600
all_tracks['time_since_midnight'].data[all_tracks['time_since_midnight'].data < 0] = np.nan
all_tracks['track_ll_rh'] = all_tracks['track_ll_rh'] * 100  # convert to percent
all_tracks['track_sfc_rh'] = all_tracks['track_sfc_rh'] * 100  # convert to percent
all_tracks['track_duration'] = (all_tracks.time - all_tracks.time.isel(timestep=0)).astype(float)/1e9/60
all_tracks['track_duration'].data[all_tracks['track_duration'] < 0] = np.nan
all_tracks['track_seabreeze_proximity'] = all_tracks['track_seabreeze_proximity'] / 1000  # convert to km
all_tracks = all_tracks.assign({
    'track_time_gradient' : (('track', 'timestep'),  np.gradient(all_tracks.track_duration, axis=1))
})
near_seabreeze_mask = (all_tracks.track_seabreeze_proximity < 10).data
near_seabreeze_time = all_tracks.track_time_gradient.data.copy()
near_seabreeze_time[~near_seabreeze_mask] = 0
near_seabreeze_time = np.nansum(near_seabreeze_time, axis=1)
all_tracks = all_tracks.assign({
    'near_seabreeze_time' : (('track',), near_seabreeze_time)
})


normalized_times = xr.open_dataset('/Volumes/LtgSSD/tobac_saves/regular_time_tracks.zarr', engine='zarr')
close_to_khgx_mask = (((normalized_times.track_projection_x_coordinate ** 2 + normalized_times.track_projection_y_coordinate ** 2)**0.5) / 1000).min(dim='time', skipna=True) <= 90
normalized_times = normalized_times.isel(track=close_to_khgx_mask)
normalized_times['track_area'] = normalized_times['track_area'] * (0.25)**2  # convert to km^2
isolated_convective_mask = (normalized_times['track_area'].max(dim='time') < 450)
normalized_times = normalized_times.isel(track=isolated_convective_mask)
normalized_times['track_present'] = ~np.isnan(normalized_times.track_seabreeze)
if seabreeze_side_select == 'all':
    seabreeze_mask = np.ones(normalized_times['track'].size, dtype=bool)
elif seabreeze_side_select == 'continental':
    seabreeze_mask = normalized_times.track_seabreeze.max(dim='time', skipna=True) == -2
elif seabreeze_side_select == 'maritime':
    seabreeze_mask = normalized_times.track_seabreeze.min(dim='time', skipna=True) == -1
elif seabreeze_side_select == 'crossing':
    seabreeze_mask = (normalized_times.track_seabreeze.min(dim='time', skipna=True) != -1) & (normalized_times.track_seabreeze.max(dim='time', skipna=True) != -2)
else:
    raise ValueError(f"Invalid seabreeze_side_select value: {seabreeze_side_select}")
normalized_times = normalized_times.isel(track=seabreeze_mask)

### Add attrs

In [ ]:
all_tracks['track_day'].attrs['long_name'] = 'Day of track'
all_tracks['track_day'].attrs['units'] = 'YYYY-MM-DD'

all_tracks['track_duration'].attrs['long_name'] = 'Track duration'
all_tracks['track_duration'].attrs['units'] = 'minutes'

# all_tracks['time'].attrs['long_name'] = 'Start time of radar volume scan'
all_tracks['time_since_midnight'].attrs['long_name'] = 'Start time of radar volume scan'
all_tracks['time_since_midnight'].attrs['units'] = 'UTC hour'

all_tracks['track_area'].attrs['long_name'] = 'Track footprint area'
all_tracks['track_area'].attrs['units'] = r'km$^2$'

all_tracks['track_ccl'].attrs['long_name'] = 'Cloud Condensation Level (CCL)'
all_tracks['track_ccl'].attrs['units'] = 'hPa'

all_tracks['track_ccn_profile_0.4'].attrs['long_name'] = 'Surface CCN concentration at 0.4% supersaturation'
all_tracks['track_ccn_profile_0.4'].attrs['units'] = 'cm$^{-3}$'


all_tracks['track_ccn_profile_0.6'].attrs['long_name'] = 'Surface CCN concentration at 0.6% supersaturation'
all_tracks['track_ccn_profile_0.6'].attrs['units'] = 'cm$^{-3}$'

all_tracks['track_child_cell_count'].attrs['long_name'] = 'Number of child cells in track'
all_tracks['track_child_cell_count'].attrs['units'] = 'count'

all_tracks['track_convT'].attrs['long_name'] = 'Convective Temperature'
all_tracks['track_convT'].attrs['units'] = '°C'

all_tracks['track_dewpoint_profile'].attrs['long_name'] = 'Surface dewpoint temperature'
all_tracks['track_dewpoint_profile'].attrs['units'] = '°C'

all_tracks['track_echotop'].attrs['long_name'] = '18 dBZ echo top height'
all_tracks['track_echotop'].attrs['units'] = 'km'

all_tracks['track_el'].attrs['long_name'] = 'Track Equilibrium Level (EL)'
all_tracks['track_el'].attrs['units'] = 'hPa'

all_tracks['track_kdpcol'].attrs['long_name'] = r'K$_{DP}$ values > 0.75 °/km, column summed, maximum over feature footprint'
all_tracks['track_kdpcol'].attrs['units'] = '°/km'

all_tracks['track_kdpcol_mean'].attrs['long_name'] = r'K$_{DP}$ values > 0.75 °/km, column summed, averaged over feature footprint'
all_tracks['track_kdpcol_mean'].attrs['units'] = '°/km'

all_tracks['track_kdpcol_total'].attrs['long_name'] = r'K$_{DP}$ values > 0.75 °/km, summed over feature volume'
all_tracks['track_kdpcol_total'].attrs['units'] = '°/km'

all_tracks['track_kdpvol'].attrs['long_name'] = r'Tobac grid cells with K$_{DP}$ > 0.75 °/km in feature footprint'
all_tracks['track_kdpvol'].attrs['units'] = 'count'

all_tracks['track_kdpwt_total'].attrs['long_name'] = r'K$_{DP}$ values > 0.75 °/km, weighted by height above the 0$^\circ$C level,'+'\ncolumn summed, total over feature footprint'
all_tracks['track_kdpwt_total'].attrs['units'] = 'm°/km'

all_tracks['track_lat'].attrs['long_name'] = 'Latitude of track centroid'
all_tracks['track_lat'].attrs['units'] = '°N'

all_tracks['track_lon'].attrs['long_name'] = 'Longitude of track centroid'
all_tracks['track_lon'].attrs['units'] = '°E'

all_tracks['track_lcl'].attrs['long_name'] = 'Lifted Condensation Level (LCL)'
all_tracks['track_lcl'].attrs['units'] = 'hPa'

all_tracks['track_lfc'].attrs['long_name'] = 'Level of Free Convection (LFC)'
all_tracks['track_lfc'].attrs['units'] = 'hPa'

all_tracks['track_max_reflectivity'].attrs['long_name'] = 'Maximum reflectivity in feature volume'
all_tracks['track_max_reflectivity'].attrs['units'] = 'dBZ'

all_tracks['track_ll_rh'].attrs['long_name'] = '0-6 km mean relative humidity'
all_tracks['track_ll_rh'].attrs['units'] = '%' 

all_tracks['track_min_L2_MCMIPC'].attrs['long_name'] = 'Minimum channel 13 brightness temperature in feature footprint'
all_tracks['track_min_L2_MCMIPC'].attrs['units'] = 'K'

all_tracks['track_mlcape'].attrs['long_name'] = '100 hPa Mixed-layer CAPE'
all_tracks['track_mlcape'].attrs['units'] = 'J/kg'

all_tracks['track_mlcin'].attrs['long_name'] = '100 hPa Mixed-layer CINH'
all_tracks['track_mlcin'].attrs['units'] = 'J/kg'

all_tracks['track_mlecape'].attrs['long_name'] = '100 hPa Mixed-layer Entrainment CAPE'
all_tracks['track_mlecape'].attrs['units'] = 'J/kg'

all_tracks['track_pressure_profile'].attrs['long_name'] = 'Surface pressure'
all_tracks['track_pressure_profile'].attrs['units'] = 'hPa'

all_tracks['track_rhvdeficitcol'].attrs['long_name'] = r'$\rho_{HV}$ deficit values > 0.02, column summed, maximum over feature footprint'
all_tracks['track_rhvdeficitcol'].attrs['units'] = 'unitless'

all_tracks['track_rhvdeficitcol_mean'].attrs['long_name'] = r'$\rho_{HV}$ deficit values > 0.02, column summed, averaged over feature footprint'
all_tracks['track_rhvdeficitcol_mean'].attrs['units'] = 'unitless'

all_tracks['track_rhvdeficitcol_total'].attrs['long_name'] = r'$\rho_{HV}$ deficit values > 0.02, summed over feature volume'
all_tracks['track_rhvdeficitcol_total'].attrs['units'] = 'unitless'

all_tracks['track_rhvdeficitvol'].attrs['long_name'] = r'Tobac grid cells with $\rho_{HV}$ deficit values > 0.02 in feature footprint'
all_tracks['track_rhvdeficitvol'].attrs['units'] = 'count'

all_tracks['track_rhvdeficitwt_total'].attrs['long_name'] = r'$\rho_{HV}$ deficit values > 0.02, weighted by height above the 0$^\circ$C level,'+'\ncolumn summed, total over feature footprint'
all_tracks['track_rhvdeficitwt_total'].attrs['units'] = 'm'

all_tracks['track_seabreeze'].attrs['long_name'] = 'Sea breeze flag'
all_tracks['track_seabreeze'].attrs['units'] = '-2 = continental, -1 = maritime'

all_tracks['track_seabreeze_proximity'].attrs['long_name'] = 'Distance from analyzed sea breeze front'
all_tracks['track_seabreeze_proximity'].attrs['units'] = 'km'

all_tracks['track_sfc_rh'].attrs['long_name'] = 'Surface relative humidity'
all_tracks['track_sfc_rh'].attrs['units'] = '%'

all_tracks['track_six_km_bwd'].attrs['long_name'] = 'Surface to 6 km bulk wind difference'
all_tracks['track_six_km_bwd'].attrs['units'] = 'm/s'

all_tracks['track_six_km_lapse'].attrs['long_name'] = 'Surface to 6 km lapse rate'
all_tracks['track_six_km_lapse'].attrs['units'] = 'K/km'

all_tracks['track_temperature_profile'].attrs['long_name'] = 'Surface temperature'
all_tracks['track_temperature_profile'].attrs['units'] = '°C'

all_tracks['track_zdrcol'].attrs['long_name'] = r'Z$_{DR}$ values > 1 dB, column summed, maximum over track footprint'
all_tracks['track_zdrcol'].attrs['units'] = 'dB'

all_tracks['track_zdrcol_mean'].attrs['long_name'] = r'Z$_{DR}$ values > 1 dB, column summed, averaged over track footprint'
all_tracks['track_zdrcol_mean'].attrs['units'] = 'dB'

all_tracks['track_zdrcol_total'].attrs['long_name'] = r'Z$_{DR}$ values > 1 dB, column summed, total over track footprint'
all_tracks['track_zdrcol_total'].attrs['units'] = 'dB'

all_tracks['track_zdrvol'].attrs['long_name'] = r'Tobac grid cells with Z$_{DR}$ > 1 dB in track footprint'
all_tracks['track_zdrvol'].attrs['units'] = 'count'

all_tracks['track_zdrwt_total'].attrs['long_name'] = r'Z$_{DR}$ values > 1 dB, weighted by height above the 0$^\circ$C level,\ncolumn summed, total over track footprint'
all_tracks['track_zdrwt_total'].attrs['units'] = 'm dB'

all_tracks['near_seabreeze_time'].attrs['long_name'] = 'Time within 10 km of sea breeze front'
all_tracks['near_seabreeze_time'].attrs['units'] = 'min'

In [ ]:
all_tracks

In [ ]:
has_kdp_mask = all_tracks.track_kdpcol.sum(dim='timestep') > 0
has_zdr_mask = all_tracks.track_zdrcol.sum(dim='timestep').data > 0
has_ltg_mask = all_tracks.track_flash_count.sum(dim='timestep').data > 0

nothing_tracks = all_tracks.isel(track=((~has_kdp_mask) & (~has_zdr_mask) & (~has_ltg_mask)))
zdr_tracks = all_tracks.isel(track=(has_zdr_mask & (~has_kdp_mask) & (~has_ltg_mask)))
kdp_tracks = all_tracks.isel(track=((~has_zdr_mask) & (has_kdp_mask) & (~has_ltg_mask)))
zdr_kdp_tracks = all_tracks.isel(track=(has_zdr_mask & has_kdp_mask & (~has_ltg_mask)))
zdr_ltg_tracks = all_tracks.isel(track=(has_zdr_mask & (~has_kdp_mask) & has_ltg_mask))
kdp_ltg_tracks = all_tracks.isel(track=((~has_zdr_mask) & has_kdp_mask & has_ltg_mask))
zdr_kdp_ltg_tracks = all_tracks.isel(track=(has_zdr_mask & has_kdp_mask & has_ltg_mask))
ltg_tracks = all_tracks.isel(track=((~has_zdr_mask) & (~has_kdp_mask) & has_ltg_mask))

has_kdp_mask = normalized_times.track_kdpcol.sum(dim='time') > 0
has_zdr_mask = normalized_times.track_zdrcol.sum(dim='time').data > 0
has_ltg_mask = normalized_times.track_flash_count.sum(dim='time').data > 0
nothing_normalized = normalized_times.isel(track=((~has_kdp_mask) & (~has_zdr_mask) & (~has_ltg_mask)))
zdr_normalized = normalized_times.isel(track=(has_zdr_mask & (~has_kdp_mask) & (~has_ltg_mask)))
kdp_normalized = normalized_times.isel(track=((~has_zdr_mask) & (has_kdp_mask) & (~has_ltg_mask)))
zdr_kdp_normalized = normalized_times.isel(track=(has_zdr_mask & has_kdp_mask & (~has_ltg_mask)))
zdr_ltg_normalized = normalized_times.isel(track=(has_zdr_mask & (~has_kdp_mask) & has_ltg_mask))
kdp_ltg_normalized = normalized_times.isel(track=((~has_zdr_mask) & has_kdp_mask & has_ltg_mask))
zdr_kdp_ltg_normalized = normalized_times.isel(track=(has_zdr_mask & has_kdp_mask & has_ltg_mask))
ltg_normalized = normalized_times.isel(track=((~has_zdr_mask) & (~has_kdp_mask) & has_ltg_mask))

In [ ]:
twoDhistDict = {
    'near_seabreeze_time' : {'funcs' : ['nothing'], 'bins' : np.arange(0, 121, 5)},
    'track_duration' : {'funcs' : ['max'], 'bins' : np.arange(0.5, 61.6, 1)},
    'time_since_midnight' : {'funcs' : ['min', 'mean'], 'bins' : np.arange(-0.5, 23.6, 1)},
    'track_area' : {'funcs' : ['max'], 'bins' : np.arange(0, 451, 10)},
    'track_ccl' : {'funcs' : ['mean'], 'bins' : np.arange(500, 1051, 25)},
    'track_ccn_profile_0.4' : {'isel' : {'vertical_levels' : 0}, 'funcs' : ['mean'], 'bins' : np.arange(0, 5001, 250)},
    'track_ccn_profile_0.6' : {'isel' : {'vertical_levels' : 0}, 'funcs' : ['mean'], 'bins' : np.arange(0, 5001, 250)},
    'track_child_cell_count' : {'funcs' : ['nothing'], 'bins' : np.arange(0.5, 10.6, 1)},
    'track_convT' : {'funcs' : ['min', 'mean', 'max'], 'bins' : np.arange(25, 41, 1)},
    'track_dewpoint_profile' : {'isel' : {'vertical_levels' : 0}, 'funcs' : ['mean'], 'bins' : np.arange(16.5, 29.6, 1)},
    'track_echotop' : {'funcs' : ['mean', 'max'], 'bins' : np.arange(0, 15.1, 0.5)},
    'track_el' : {'funcs' : ['mean'], 'bins' : np.arange(100, 301, 10)},
    'track_kdpcol' : {'funcs' : ['sum', 'mean', 'max'], 'scale' : 'xlog', 'bins' : [np.logspace(0, 1.84, 10), np.logspace(0, 1.1, 10), np.logspace(0, 1.3, 10)]}, # sum of values vertical dimension (in 3 km slab) above Kdp threshold, and then max anywhere in 2D feature. "Column strength."
    'track_kdpcol_mean' : {'funcs' : ['sum', 'mean', 'max'], 'scale' : 'xlog', 'bins' : [np.logspace(-6, -2.5, 16), np.logspace(-6, -3, 16), np.logspace(-6, -3, 16)]}, # sum of values vertical dimension (in 3 km slab) above Zdr threshold, and then average across 2D feature.    
    'track_kdpcol_total' : {'funcs' : ['sum', 'mean', 'max'], 'bins' : [np.arange(0, 20001, 1000), np.arange(0, 5001, 100), np.arange(0, 5001, 100)]}, # sum of values in vertical and horizontal dimensions above Kdp threshold over the whole 3D column feature.
    'track_kdpvol' : {'funcs' : ['sum', 'mean', 'max'], 'scale' : 'xlog', 'bins' : [np.logspace(0, 2.5, 16), np.logspace(0, 2, 16), np.logspace(0, 2, 16)]}, # count of grid boxes Kdp above 0.75 dB, in 3 km slab above melting level in feature. Thresholds from van Lier Walqui.
    'track_kdpwt_total' : {'funcs' : ['sum', 'mean', 'max'], 'bins' : [np.arange(0, 2.001e7, 5e5), np.arange(0, 2.001e6, 5e4), np.arange(0, 5.001e6, 1e5)]}, # the total Kdp values in the feature, occurring between the melting level and 1km below the freezing level, weighted by its height above the melting level.
    'track_lat' : {'funcs' : ['mean'], 'bins' : np.arange(26, 33.1, 0.1)},
    'track_lcl' : {'funcs' : ['mean', 'min'], 'bins' : np.arange(650, 1051, 10)},
    'track_lfc' : {'funcs' : ['mean', 'min'], 'bins' : np.arange(500, 1051, 10)},
    'track_ll_rh' : {'funcs' : ['mean', 'max'], 'bins' : np.arange(30, 100.1, 1)},
    'track_lon' : {'funcs' : ['mean'], 'bins' : np.arange(-98, -91.9, 0.1)},
    'track_max_reflectivity' : {'funcs' : ['mean', 'max'], 'bins' : np.arange(15, 76, 1)},
    'track_min_L2_MCMIPC' : {'funcs' : ['min'], 'bins' : np.arange(190, 301, 2)},
    'track_mlcape' : {'funcs' : ['mean', 'max'], 'bins' : np.arange(0, 4000, 50)},
    'track_mlcin' : {'funcs' : ['mean', 'min'], 'bins' : np.arange(-250, 1, 5)},
    'track_mlecape' : {'funcs' : ['mean', 'max'], 'bins' : np.arange(0, 3500, 50)},
    'track_pressure_profile' : {'isel' : {'vertical_levels' : 0}, 'funcs' : ['mean'], 'bins' : np.arange(990, 1026, 2.5)},
    'track_rhvdeficitcol' : {'funcs' : ['sum', 'mean', 'max'], 'bins' : [np.arange(0, 50.1, 0.25), np.arange(0, 2.501, 0.1), np.arange(0, 0.601, 0.05)]},
    'track_rhvdeficitcol_mean' : {'funcs' : ['sum', 'mean', 'max'], 'bins' : [np.arange(0, 0.02, 0.0005), np.arange(0, 0.007501, 0.0001), np.arange(0, 0.00201, 0.00005)]},
    'track_rhvdeficitcol_total' : {'funcs' : ['sum', 'mean', 'max'], 'bins' : np.arange(0, 4001, 50)},
    'track_rhvdeficitvol' : {'funcs' : ['sum', 'mean', 'max'], 'bins' : np.arange(0, 4001, 50)},
    'track_rhvdeficitwt_total' : {'funcs' : ['sum', 'mean', 'max'], 'bins' : np.arange(0, 10001, 50)},
    'track_seabreeze' : {'funcs' : ['mean'], 'bins' : np.arange(-2.1, -0.9, 0.1)},
    'track_seabreeze_proximity' : {'funcs' : ['mean', 'min'], 'bins' : np.arange(0, 41, 1)},
    'track_sfc_rh' : {'funcs' : ['mean', 'max'], 'bins' : np.arange(30, 101, 5)},
    'track_six_km_bwd' : {'funcs' : ['mean', 'max'], 'bins' : np.arange(0, 25, 1)},
    'track_six_km_lapse' : {'funcs' : ['mean', 'max'], 'bins': np.arange(-8, -3.9, 0.2)},
    'track_temperature_profile' : {'isel' : {'vertical_levels' : 0}, 'funcs' : ['mean'], 'bins' : np.arange(24, 41, 1)},
    'track_zdrcol' : {'funcs' : ['sum', 'mean', 'max'], 'bins' : np.arange(0, 71, 1)},
    'track_zdrcol_mean' : {'funcs' : ['sum', 'mean', 'max'], 'scale' : 'xlog', 'bins' : [np.logspace(-5, -2, 50), np.logspace(-6, -2.5, 50), np.logspace(-5, -2, 50)]},
    'track_zdrcol_total' : {'funcs' : ['sum', 'mean', 'max'], 'bins' : np.arange(0, 2001, 50)},
    'track_zdrvol' : {'funcs' : ['sum', 'mean', 'max'], 'scale' : 'xlog', 'bins' : [np.logspace(0, 3, 50), np.logspace(0, 2.5, 50), np.logspace(0, 2.5, 50)]},
    'track_zdrwt_total' : {'funcs' : ['sum', 'mean', 'max'], 'bins' : np.arange(0, 2001, 50)},
}

In [ ]:
def calculate_error_ellipse(x, y, confidence_interval=0.68):
    """Calculate parameters for an error ellipse representing the covariance of x and y data.
    
    Parameters
    ----------
    x : array-like
        1D array of x data points.
    y : array-like
        1D array of y data points.
    confidence_interval : float, optional
        Confidence interval for the ellipse (default is 0.68 for 68% confidence).
    
    Returns
    -------
    mu_x : float
        Mean of x data points.
    mu_y : float
        Mean of y data points.
    ell_x : np.ndarray
        Lengths of the semi-major and semi-minor axes of the ellipse.
    angle : float
        Rotation angle of the ellipse in degrees.
    """
    chi2_val = chi2.ppf(confidence_interval, 2)
    mu_x, mu_y = np.mean(x), np.mean(y)
    eig = np.linalg.eig(np.cov(x, y))
    lambda_x = eig.eigenvalues
    nu_x_2, nu_x_1 = eig.eigenvectors
    ell_x = 2*(chi2_val*lambda_x)**0.5
    angle = np.rad2deg(np.arctan2(nu_x_1[0], nu_x_1[1]))
    return mu_x, mu_y, ell_x, angle

### 2D histograms

In [ ]:
flash_count = all_tracks.track_flash_count.sum(dim='timestep', skipna=True).data
flash_count[flash_count == 0] = 1e-9
for dv, things_to_do in twoDhistDict.items():
    for j, this_func in enumerate(things_to_do['funcs']):
        thing_to_plot = all_tracks[dv]
        if 'isel' in things_to_do:
            thing_to_plot = thing_to_plot.isel(**things_to_do['isel'])
        if this_func == 'mean':
            thing_to_plot = thing_to_plot.mean(dim='timestep', skipna=True)
            fancy_string = ', mean along track'
        elif this_func == 'min':
            thing_to_plot = thing_to_plot.min(dim='timestep', skipna=True)
            fancy_string = ', minimum along track'
        elif this_func == 'max':
            thing_to_plot = thing_to_plot.max(dim='timestep', skipna=True)
            fancy_string = ', maximum along track'
        elif this_func == 'sum':
            thing_to_plot = thing_to_plot.sum(dim='timestep', skipna=True)
            fancy_string = ', sum along track'
        elif this_func == 'nothing':
            fancy_string = ''
        else:
            raise ValueError(f"Unknown function {this_func} for variable {dv}")
        if 'time' in thing_to_plot.name:
            thing_to_plot = thing_to_plot.data.astype(float)
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(1, 1, 1)
        if type(things_to_do['bins']) == np.ndarray:
            x_bins = things_to_do['bins'].copy()
        elif type(things_to_do['bins']) == list:
            x_bins = np.array(things_to_do['bins'][j]).copy()
        else:
            raise ValueError(f"Unknown bins type {type(things_to_do['bins'])} for variable {dv}")
        x_bin0 = x_bins[0]
        x_binf = x_bins[-1]
        if things_to_do.get('scale', None) is not None:
            if 'xlog' in things_to_do['scale']:
                thing_to_plot = thing_to_plot.copy()
                thing_to_plot[thing_to_plot == 0] = 1e-11
                x_bins = np.append([1e-9], x_bins)
        x_bins[0] = np.min([thing_to_plot.min(), x_bin0])
        x_bins[-1] = np.max([thing_to_plot.max(), x_binf])
        y_bins = np.logspace(0, 3, 100)
        y_bins = np.append([1e-8], y_bins)
        y_bin0 = y_bins[0]
        y_binf = y_bins[-1]
        y_bins[0] = np.min([flash_count.min(), y_bin0])
        y_bins[-1] = np.max([flash_count.max(), y_binf])
        art = ax.hist2d(
            thing_to_plot,
            flash_count,
            cmap='viridis', norm=pltcolors.LogNorm(), bins=[x_bins, y_bins], range=[[thing_to_plot.min(), thing_to_plot.max()], [flash_count.min(), flash_count.max()]], rasterized=True,
        )
        thing_to_plot_nonnan = thing_to_plot[~np.isnan(thing_to_plot) & ~np.isnan(flash_count)]
        flash_count_nonnan = flash_count[~np.isnan(thing_to_plot) & ~np.isnan(flash_count)]
        ax.set_yscale('log')
        if things_to_do.get('scale', None) is not None:
            if 'xlog' in things_to_do['scale']:
                ax.set_xscale('log')
                x_bin0 -= 10**int((np.log10(x_bin0) - 1))
        ax.set_xlim(x_bin0, x_binf)
        ax.set_ylim(0.9, y_binf)
        ax.set_xlabel(f'{all_tracks[dv].attrs.get('long_name', dv)}{fancy_string} ({all_tracks[dv].attrs.get('units', '')})')
        ax.set_ylabel('Track Flash Count')
        fig.colorbar(art[3], ax=ax, label='Counts')
        if seabreeze_side_select != 'all':
            ax.set_title(f'{seabreeze_side_select} tracks')
        fig.tight_layout()
        fig.savefig(f'./thesis_figs/{seabreeze_side_select}/2dhists/{dv}--{this_func}.pdf')
        mu_x, mu_y, ell_x, angle = calculate_error_ellipse(thing_to_plot_nonnan, flash_count_nonnan)
        ellipse = Ellipse((mu_x, mu_y), width=ell_x[0], height=ell_x[1], angle=angle,
                          fill=False, color='k', linewidth=3, alpha=0.5, label=r'1-$\sigma$ confidence interval (all)')
        ax.add_patch(ellipse)
        ax.scatter(mu_x, mu_y, color='k', marker='P', s=40, linewidths=0.75, edgecolors='white')
        mu_x_ltg, mu_y_ltg, ell_x_ltg, angle_ltg = calculate_error_ellipse(thing_to_plot_nonnan[flash_count_nonnan >= 1], flash_count_nonnan[flash_count_nonnan >= 1])
        ellipse_ltg = Ellipse((mu_x_ltg, mu_y_ltg), width=ell_x_ltg[0], height=ell_x_ltg[1], angle=angle_ltg,
                              fill=False, color='r', linewidth=3, alpha=0.5, label=r'1-$\sigma$ confidence interval (lightning > 0)')
        ax.add_patch(ellipse_ltg)
        ax.scatter(mu_x_ltg, mu_y_ltg, color='r', marker='P', s=40, linewidths=0.75, edgecolors='white')
        try:
            reg_base = np.linspace(x_bin0, x_binf, 1000)
            this_reg = linregress(thing_to_plot_nonnan, flash_count_nonnan)
            this_pval_str = f'={this_reg.pvalue:.3f}' if this_reg.pvalue >= 0.001 else f'<.001'
            ax.plot(reg_base, this_reg.intercept + this_reg.slope*reg_base, color='k', linestyle='--', label=f'y={this_reg.slope:.2f}x+{this_reg.intercept:.1f}\n'+r'r$^{2}$'+f'={this_reg.rvalue**2:.2f}, p{this_pval_str}')
            ltg_reg = linregress(thing_to_plot_nonnan[flash_count_nonnan >= 1], flash_count_nonnan[flash_count_nonnan >= 1])
            ltg_pval_str = f'={ltg_reg.pvalue:.3f}' if ltg_reg.pvalue >= 0.001 else f'<.001'
            ax.plot(reg_base, ltg_reg.intercept + ltg_reg.slope*reg_base, color='r', linestyle='--', label=f'y={ltg_reg.slope:.2f}x+{ltg_reg.intercept:.1f}\n'+r'r$^{2}$'+f'={ltg_reg.rvalue**2:.2f}, p{ltg_pval_str}')
        except ValueError:
            print(f"Skipping regression for {dv} with {this_func} due to insufficient data.")
        ax.legend()
        fig.savefig(f'./thesis_figs/{seabreeze_side_select}/2dhists_ellipse/{dv}--{this_func}.pdf')
        plt.close(fig)

### Error Ellipses

In [ ]:
styles_dict = {
    'nothing' : {'color' : 'tab:blue', 'label' : 'Nothing', 'alpha' : 0.2, 'linewidth' : 1},
    'zdr' : {'color' : 'tab:orange', 'label' : r'Z$_{DR}$', 'alpha' : 0.2, 'linewidth' : 1},
    'kdp' : {'color' : 'tab:green', 'label' : r'K$_{DP}$', 'alpha' : 0.2, 'linewidth' : 1},
    'zdr_kdp' : {'color' : 'tab:red', 'label' : r'Z$_{DR}$ K$_{DP}$', 'alpha' : 0.2, 'linewidth' : 1},
    'zdr_ltg' : {'color' : 'tab:purple', 'label' : r'Z$_{DR}$ Lightning', 'alpha' : 0.2, 'linewidth' : 1},
    'kdp_ltg' : {'color' : 'tab:brown', 'label' : r'K$_{DP}$ Lightning', 'alpha' : 0.2, 'linewidth' : 1},
    'zdr_kdp_ltg' : {'color' : 'tab:pink', 'label' : r'Z$_{DR}$ K$_{DP}$ Lightning', 'alpha' : 0.2, 'linewidth' : 1},
    'ltg' : {'color' : 'tab:gray', 'label' : 'Lightning', 'alpha' : 0.2, 'linewidth' : 1}
}
flash_counts_dict = {
    'nothing' : nothing_tracks.track_flash_count.sum(dim='timestep', skipna=True),
    'zdr' : zdr_tracks.track_flash_count.sum(dim='timestep', skipna=True),
    'kdp' : kdp_tracks.track_flash_count.sum(dim='timestep', skipna=True),
    'zdr_kdp' : zdr_kdp_tracks.track_flash_count.sum(dim='timestep', skipna=True),
    'zdr_ltg' : zdr_ltg_tracks.track_flash_count.sum(dim='timestep', skipna=True),
    'kdp_ltg' : kdp_ltg_tracks.track_flash_count.sum(dim='timestep', skipna=True),
    'zdr_kdp_ltg' : zdr_kdp_ltg_tracks.track_flash_count.sum(dim='timestep', skipna=True),
    'ltg' : ltg_tracks.track_flash_count.sum(dim='timestep', skipna=True)
}

### 1D histograms

In [ ]:
for dv, things_to_do in twoDhistDict.items():
    if things_to_do.get('scale', None) is not None:
        if 'xlog' in things_to_do['scale']:
            should_xlog = True
        else:
            should_xlog = False
    else:
        should_xlog = False
    for j, this_func in enumerate(things_to_do['funcs']):
        things_to_plot = {
            'nothing' : nothing_tracks[dv],
            'zdr' : zdr_tracks[dv],
            'kdp' : kdp_tracks[dv],
            'zdr_kdp' : zdr_kdp_tracks[dv],
            'zdr_ltg' : zdr_ltg_tracks[dv],
            'kdp_ltg' : kdp_ltg_tracks[dv],
            'zdr_kdp_ltg' : zdr_kdp_ltg_tracks[dv],
            'ltg' : ltg_tracks[dv],
        }
        if 'isel' in things_to_do:
            for key, val in things_to_plot.items():
                things_to_plot[key] = val.isel(**things_to_do['isel'])
        if this_func == 'mean':
            for key, val in things_to_plot.items():
                things_to_plot[key] = val.mean(dim='timestep', skipna=True)
            fancy_string = ',\nmean along track'
        elif this_func == 'min':
            for key, val in things_to_plot.items():
                things_to_plot[key] = val.min(dim='timestep', skipna=True)
            fancy_string = ',\nminimum along track'
        elif this_func == 'max':
            for key, val in things_to_plot.items():
                things_to_plot[key] = val.max(dim='timestep', skipna=True)
            fancy_string = ',\nmaximum along track'
        elif this_func == 'sum':
            for key, val in things_to_plot.items():
                things_to_plot[key] = val.sum(dim='timestep', skipna=True)
            fancy_string = ',\nsum along track'
        elif this_func == 'nothing':
            fancy_string = ''
        else:
            raise ValueError(f"Unknown function {this_func} for variable {dv}")
        valmin = np.inf
        valmax = -np.inf
        if 'time' in things_to_plot['nothing'].name:
            for key, val in things_to_plot.items():
                things_to_plot[key] = val.data.astype(float)
        fig, axs = plt.subplots(nrows=3, ncols=2, figsize=(8.1, 8))
        axs = axs.flatten()
        if type(things_to_do['bins']) == np.ndarray:
            x_bins = things_to_do['bins'].copy()
        elif type(things_to_do['bins']) == list:
            x_bins = np.array(things_to_do['bins'][j]).copy()
        else:
            raise ValueError(f"Unknown bins type {type(things_to_do['bins'])} for variable {dv}")
        x_bin0 = x_bins[0]
        x_binf = x_bins[-1]
        x_bins[0] = np.min([np.nanmin(np.hstack(list(things_to_plot.values()))), x_bin0])
        x_bins[-1] = np.max([np.nanmax(np.hstack(list(things_to_plot.values()))), x_binf])
        axs[0].hist(things_to_plot.values(), histtype='barstacked', label=[styles_dict[key]['label'] for key in things_to_plot.keys()],
                color=[styles_dict[key]['color'] for key in things_to_plot.keys()], bins=x_bins)
        histos = np.array([np.histogram(t, bins=x_bins)[0] for t in things_to_plot.values()])
        histo_fracs = (histos/np.sum(histos, axis=0))*100
        histo_fracs[np.isnan(histo_fracs)] = 0
        hist_handles = [axs[1].stairs(np.sum(histo_fracs[0:i+1], axis=0), x_bins, baseline=np.sum(histo_fracs[0:i], axis=0), label=styles_dict[key]['label'], color=styles_dict[key]['color'], linewidth=1, fill=True) for i, key in enumerate(things_to_plot.keys())]
        x_bin_ctr = (x_bins[:-1] + x_bins[1:])/2
        
        invalid_histo_mask = np.sum(histos, axis=0) > 30
        histo_fracs = histo_fracs[:, invalid_histo_mask]
        x_bin_ctr = x_bin_ctr[invalid_histo_mask]
        [axs[3].scatter(x_bin_ctr, histo_fracs[i], color=styles_dict[key]['color'], s=5) for i, key in enumerate(things_to_plot.keys())]
        regrs = [linregress(x_bin_ctr, histo_fracs[i]) for i in range(histo_fracs.shape[0])]
        if should_xlog:
            regrs_log = [linregress(np.log10(x_bin_ctr), histo_fracs[i]) for i in range(histo_fracs.shape[0])]
            regrs_pval_str = [f'={regrs_log[i].pvalue:.3f}' if regrs_log[i].pvalue >= 0.001 else f'<.001' for i in range(len(regrs_log))]
            regrs_labels = [f'm={regrs_log[i].slope:.2e} %/{all_tracks[dv].attrs.get('units', '')}\n'+
                            f'b={regrs_log[i].intercept:.0f}%\n'+r'r$^{2}$'+f'={regrs_log[i].rvalue**2:.2f}, p{regrs_pval_str[i]}' 
                            if not np.isnan(regrs_log[i].rvalue) else '(no tracks)\n' for i in range(len(regrs_log))]
        else:
            regrs_pval_str = [f'={regrs[i].pvalue:.3f}' if regrs[i].pvalue >= 0.001 else f'<.001' for i in range(len(regrs))]
            regrs_labels = [f'm={regrs[i].slope:.2e} %/{all_tracks[dv].attrs.get('units', '')}\n'+
                            f'b={regrs[i].intercept:.0f}%\n'+r'r$^{2}$'+f'={regrs[i].rvalue**2:.2f}, p{regrs_pval_str[i]}' 
                            if not np.isnan(regrs[i].rvalue) else '(no tracks)\n' for i in range(len(regrs))]
        reg_handles = [axs[3].plot(x_bin_ctr, regrs[i].intercept + regrs[i].slope*x_bin_ctr, color=styles_dict[key]['color'], linestyle='--', label=regrs_labels[i]) for i, key in enumerate(things_to_plot.keys())]
        axs[3].set_ylabel('Percentage of Tracks')
        nonltg = np.zeros(histo_fracs.shape[1], dtype=float)
        ltg = np.zeros(histo_fracs.shape[1], dtype=float)
        for i, histo_frac in enumerate(histo_fracs):
            if 'Lightning' in styles_dict[list(things_to_plot.keys())[i]]['label']:
                ltg += histo_frac
            else:
                nonltg += histo_frac
        invalid_mask = (nonltg + ltg) == 0
        x_bin_ctr = x_bin_ctr[~invalid_mask]
        nonltg = nonltg[~invalid_mask]
        ltg = ltg[~invalid_mask]
        nonltg_pts = axs[5].scatter(x_bin_ctr, nonltg, color='k', s=5, label='All non-lightning')
        ltg_pts = axs[5].scatter(x_bin_ctr, ltg, color='tab:olive', s=5, label='All lightning')

        nonltg_regr = linregress(x_bin_ctr, nonltg)
        ltg_regr = linregress(x_bin_ctr, ltg)
        if should_xlog:
            regr_log_nonltg = linregress(np.log10(x_bin_ctr), nonltg)
            regr_log_ltg = linregress(np.log10(x_bin_ctr), ltg)
            regr_label_nonltg = f'm={regr_log_nonltg.slope:.2e} %/{all_tracks[dv].attrs.get('units', '')}\n'
            nonltg_pval_str = f'={regr_log_nonltg.pvalue:.3f}' if regr_log_nonltg.pvalue >= 0.001 else f'<.001'
            regr_label_nonltg += f'b={regr_log_nonltg.intercept:.0f}%\n'+r'r$^{2}$'+f'={regr_log_nonltg.rvalue**2:.2f}, p{nonltg_pval_str}'
            regr_label_ltg = f'm={regr_log_ltg.slope:.2e} %/{all_tracks[dv].attrs.get('units', '')}\n'
            ltg_pval_str = f'={regr_log_ltg.pvalue:.3f}' if regr_log_ltg.pvalue >= 0.001 else f'<.001'
            regr_label_ltg += f'b={regr_log_ltg.intercept:.0f}%\n'+r'r$^{2}$'+f'={regr_log_ltg.rvalue**2:.2f}, p{ltg_pval_str}'
        else:
            regr_label_nonltg = f'm={nonltg_regr.slope:.2e} %/{all_tracks[dv].attrs.get('units', '')}\n'
            nonltg_pval_str = f'={nonltg_regr.pvalue:.3f}' if nonltg_regr.pvalue >= 0.001 else f'<.001'
            regr_label_nonltg += f'b={nonltg_regr.intercept:.0f}%\n'+r'r$^{2}$'+f'={nonltg_regr.rvalue**2:.2f}, p{nonltg_pval_str}'
            regr_label_ltg = f'm={ltg_regr.slope:.2e} %/{all_tracks[dv].attrs.get('units', '')}\n'
            ltg_pval_str = f'={ltg_regr.pvalue:.3f}' if ltg_regr.pvalue >= 0.001 else f'<.001'
            regr_label_ltg += f'b={ltg_regr.intercept:.0f}%\n'+r'r$^{2}$'+f'={ltg_regr.rvalue**2:.2f}, p{ltg_pval_str}'
        # if divideby != 1:
        #     regr_label_nonltg = regr_label_nonltg.replace('x', f'x/{divideby}')
        #     regr_label_ltg = regr_label_ltg.replace('x', f'x/{divideby}')
        nonltg_regr_handles = axs[5].plot(x_bin_ctr, nonltg_regr.intercept + nonltg_regr.slope*x_bin_ctr, color='k', linestyle='dashdot', label=regr_label_nonltg)
        ltg_regr_handles = axs[5].plot(x_bin_ctr, ltg_regr.intercept + ltg_regr.slope*x_bin_ctr, color='tab:olive', linestyle='dashdot', label=regr_label_ltg)

        [ax.set_xlim(x_bin0, x_binf) for ax in axs]

        [ax.set_xlabel(f'{all_tracks[dv].attrs.get('long_name', dv).replace('brightness', 'brightness\n').replace('summed, averaged',
                        'summed,\naveraged').replace('km in fea', 'km\n in fea')}{fancy_string} ({all_tracks[dv].attrs.get('units', '')})') for ax in axs]
        axs[5].set_xlabel(f'{axs[5].get_xlabel()}\nBins with statistically small sample size\n(< 30 tracks) not included in regression.')
        axs[0].set_title('Track Counts')
        axs[1].set_title('Track Percentages')
        axs[0].set_ylabel('Track Count')
        [ax.set_ylabel('Percentage of tracks') for ax in axs[1:]]
        [ax.set_ylim(0, 100) for ax in axs[1:]]
        if should_xlog:
            for ax in axs:
                ax.set_xscale('log')
                axs[0].set_yscale('log')

        cat_handles = hist_handles
        cat_handles.extend([nonltg_pts, ltg_pts])
        reg_handles = [r[0] for r in reg_handles]
        reg_handles.extend([nonltg_regr_handles[0], ltg_regr_handles[0]])

        axs[2].legend(handles=cat_handles, labels=[h.get_label() for h in cat_handles], loc='upper center', ncols=2)
        axs[2].axis('off')
        axs[4].legend(handles=reg_handles, labels=[h.get_label() for h in reg_handles], loc='upper center', ncols=2)
        axs[4].axis('off')
        if should_xlog:
            axs[4].set_title('Log-Linear Regression (y = m log10(x) + b)')
        else:
            axs[4].set_title('Linear Regression (y = mx + b)')
        if seabreeze_side_select != 'all':
            if seabreeze_side_select == 'crossing':
                fig.suptitle('Tracks crossing the seabreeze front')
            else:
                fig.suptitle(f'{seabreeze_side_select} tracks'.title())
        fig.tight_layout()
        ax4pos = axs[4].get_position()
        axs[4].set_position([ax4pos.x0, ax4pos.y0 + 0.09, ax4pos.width, ax4pos.height])
        fig.savefig(f'./thesis_figs/{seabreeze_side_select}/categorical_hists/{dv}--{this_func}.pdf')
        plt.close(fig)

### Along-Track times

In [ ]:
cutoff = 30
things_to_plot = {
    'nothing' : nothing_normalized.track_present.sum(dim='track'),
    'zdr' : zdr_normalized.track_present.sum(dim='track'),
    'kdp' : kdp_normalized.track_present.sum(dim='track'),
    'zdr_kdp' : zdr_kdp_normalized.track_present.sum(dim='track'),
    'zdr_ltg' : zdr_ltg_normalized.track_present.sum(dim='track'),
    'kdp_ltg' : kdp_ltg_normalized.track_present.sum(dim='track'),
    'zdr_kdp_ltg' : zdr_kdp_ltg_normalized.track_present.sum(dim='track'),
    'ltg' : ltg_normalized.track_present.sum(dim='track')
}
normalized_times.track_present.sum(dim='track')
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
for ax in axs:
    [ax.plot(normalized_times.time/60, things_to_plot[key], label=styles_dict[key]['label'], color=styles_dict[key]['color'], linewidth=1) for key in styles_dict.keys()]
    ax.plot(normalized_times.time/60, normalized_times.track_present.sum(dim='track'), color='k')
    ax.set_xlabel('Time since track start (minutes)')
    ax.set_ylabel('Number of tracks contributing')
    ax.axvline(cutoff, color='r', linestyle='--', label=f'{cutoff} min cutoff')
axs[0].legend()
axs[1].set_yscale('log')
[ax.set_ylim(0, 7500) for ax in axs]
axs[2].set_xlim(0, cutoff)

fig.suptitle('Normalized track time - contributing tracks over time')
fig.savefig(f'./thesis_figs/normalized_track_times.pdf')
plt.close(fig)

In [ ]:
for dv, things_to_do in twoDhistDict.items():
    if dv not in  zdr_kdp_ltg_normalized.data_vars:
        continue
    everything_mu = zdr_kdp_ltg_normalized[dv].mean(dim='track', skipna=True)
    everything_sigma = zdr_kdp_ltg_normalized[dv].std(dim='track', skipna=True)
    zdr_mu = zdr_normalized[dv].mean(dim='track', skipna=True)
    zdr_sigma = zdr_normalized[dv].std(dim='track', skipna=True)
    zdr_kdp_mu = zdr_kdp_normalized[dv].mean(dim='track', skipna=True)
    zdr_kdp_sigma = zdr_kdp_normalized[dv].std(dim='track', skipna=True)
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(1, 1, 1)
    ax.plot(everything_mu.time/60, everything_mu, color='tab:pink', label=r'Z$_{DR}$ K$_{DP}$ Lightning')
    ax.plot(everything_mu.time/60, everything_mu - everything_sigma, color='tab:pink', ls='--')
    ax.plot(everything_mu.time/60, everything_mu + everything_sigma, color='tab:pink', ls='--')
    ax.fill_between(everything_mu.time/60, everything_mu - everything_sigma, everything_mu + everything_sigma, color='tab:pink', alpha=0.1, label=r'1-$\sigma$ interval')
    if 'track_kdp' in dv:
        significance = ttest_ind(zdr_kdp_ltg_normalized[dv], zdr_kdp_normalized[dv], axis=0, nan_policy='omit').pvalue < 0.05
        ax.scatter(everything_mu.time[significance]/60, everything_mu[significance], color='k', s=5)
        ax.plot(zdr_kdp_mu.time/60, zdr_kdp_mu, color='tab:red', label=r'Z$_{DR}$ K$_{DP}$')
        ax.plot(zdr_kdp_mu.time/60, zdr_kdp_mu - zdr_kdp_sigma, color='tab:red', ls='--')
        ax.plot(zdr_kdp_mu.time/60, zdr_kdp_mu + zdr_kdp_sigma, color='tab:red', ls='--')
        ax.fill_between(zdr_kdp_mu.time/60, zdr_kdp_mu - zdr_kdp_sigma, zdr_kdp_mu + zdr_kdp_sigma, color='tab:red', alpha=0.1, label=r'1-$\sigma$ interval')
        ax.scatter(zdr_kdp_mu.time[significance]/60, zdr_kdp_mu[significance], color='k', s=5, label='Times with statistically significant difference in distributions (t-test, p < 0.05)')
    else:
        significance = ttest_ind(zdr_kdp_ltg_normalized[dv], zdr_normalized[dv], axis=0, nan_policy='omit').pvalue < 0.05
        ax.scatter(everything_mu.time[significance]/60, everything_mu[significance], color='k', s=5)
        ax.plot(zdr_mu.time/60, zdr_mu, color='tab:orange', label=r'Z$_{DR}$')
        ax.plot(zdr_mu.time/60, zdr_mu - zdr_sigma, color='tab:orange', ls='--')
        ax.plot(zdr_mu.time/60, zdr_mu + zdr_sigma, color='tab:orange', ls='--')
        ax.fill_between(zdr_mu.time/60, zdr_mu - zdr_sigma, zdr_mu + zdr_sigma, color='tab:orange', alpha=0.1, label=r'1-$\sigma$ interval')
        ax.scatter(zdr_mu.time[significance]/60, zdr_mu[significance], color='k', s=5, label='Times with statistically significant difference in distributions (t-test, p < 0.05)')
    ax.set_xlabel('Time (minutes since start of track)')
    ax.set_ylabel(f'{all_tracks[dv].attrs.get("long_name", dv)} ({all_tracks[dv].attrs.get("units", "")})')
    ax.legend()
    ax.set_xlim(0, 30)
    if type(things_to_do['bins']) == np.ndarray:
        x_bins = things_to_do['bins'].copy()
    elif type(things_to_do['bins']) == list:
        x_bins = np.array(things_to_do['bins'][j]).copy()
    else:
        raise ValueError(f"Unknown bins type {type(things_to_do['bins'])} for variable {dv}")
    x_bin0 = x_bins[0]
    x_binf = x_bins[-1]
    x_bins[0] = np.min([np.nanmin(np.hstack(list(things_to_plot.values()))), x_bin0])
    x_bins[-1] = np.max([np.nanmax(np.hstack(list(things_to_plot.values()))), x_binf])
    if seabreeze_side_select != 'all':
        ax.set_title(f'{seabreeze_side_select} tracks')
    fig.savefig(f'./thesis_figs/{seabreeze_side_select}/along_track_duration_comp/{dv}.pdf')
    plt.close(fig)

### Track Fractions for all days

In [ ]:
track_counts_fig = plt.figure(figsize=(8.5, 8.5))
counts_gs = GridSpec(3, 2, figure=track_counts_fig, height_ratios=[1, 1, 1e-4])
axs_counts = [track_counts_fig.add_subplot(counts_gs[i, j]) for i in range(2) for j in range(2)]
axs_counts.append(track_counts_fig.add_subplot(counts_gs[2, :]))
track_fracks_fig = plt.figure(figsize=(8.5, 8.5))
fracks_gs = GridSpec(3, 2, figure=track_fracks_fig, height_ratios=[1, 1, 1e-4])
axs_fracks = [track_fracks_fig.add_subplot(fracks_gs[i, j]) for i in range(2) for j in range(2)]
axs_fracks.append(track_fracks_fig.add_subplot(fracks_gs[2, :]))
for j, seabreeze_side_select2 in enumerate(['all', 'continental', 'maritime', 'crossing']):
    unique_days = np.sort(np.unique(all_tracks['track_day'].data))
    nothing_sum = np.zeros(unique_days.shape, dtype=int)
    zdr_sum = np.zeros(unique_days.shape, dtype=int)
    kdp_sum = np.zeros(unique_days.shape, dtype=int)
    zdr_kdp_sum = np.zeros(unique_days.shape, dtype=int)
    zdr_ltg_sum = np.zeros(unique_days.shape, dtype=int)
    kdp_ltg_sum = np.zeros(unique_days.shape, dtype=int)
    zdr_kdp_ltg_sum = np.zeros(unique_days.shape, dtype=int)
    ltg_sum = np.zeros(unique_days.shape, dtype=int)
    rolling_sum = np.zeros(unique_days.shape, dtype=int)
    track_counts_ax = axs_counts[j]
    track_fracks_ax = axs_fracks[j]
    if seabreeze_side_select2 == 'all':
        seabreeze_mask_nothing = np.ones(nothing_tracks['track'].size, dtype=bool)
        seabreeze_mask_zdr = np.ones(zdr_tracks['track'].size, dtype=bool)
        seabreeze_mask_kdp = np.ones(kdp_tracks['track'].size, dtype=bool)
        seabreeze_mask_zdr_kdp = np.ones(zdr_kdp_tracks['track'].size, dtype=bool)
        seabreeze_mask_zdr_ltg = np.ones(zdr_ltg_tracks['track'].size, dtype=bool)
        seabreeze_mask_kdp_ltg = np.ones(kdp_ltg_tracks['track'].size, dtype=bool)
        seabreeze_mask_zdr_kdp_ltg = np.ones(zdr_kdp_ltg_tracks['track'].size, dtype=bool)
        seabreeze_mask_ltg = np.ones(ltg_tracks['track'].size, dtype=bool)
    elif seabreeze_side_select2 == 'continental':
        seabreeze_mask_nothing = nothing_tracks.track_seabreeze.max(dim='timestep', skipna=True) == -2
        seabreeze_mask_zdr = zdr_tracks.track_seabreeze.max(dim='timestep', skipna=True) == -2
        seabreeze_mask_kdp = kdp_tracks.track_seabreeze.max(dim='timestep', skipna=True) == -2
        seabreeze_mask_zdr_kdp = zdr_kdp_tracks.track_seabreeze.max(dim='timestep', skipna=True) == -2
        seabreeze_mask_zdr_ltg = zdr_ltg_tracks.track_seabreeze.max(dim='timestep', skipna=True) == -2
        seabreeze_mask_kdp_ltg = kdp_ltg_tracks.track_seabreeze.max(dim='timestep', skipna=True) == -2
        seabreeze_mask_zdr_kdp_ltg = zdr_kdp_ltg_tracks.track_seabreeze.max(dim='timestep', skipna=True) == -2
        seabreeze_mask_ltg = ltg_tracks.track_seabreeze.max(dim='timestep', skipna=True) == -2
    elif seabreeze_side_select2 == 'maritime':
        seabreeze_mask_nothing = nothing_tracks.track_seabreeze.min(dim='timestep', skipna=True) == -1
        seabreeze_mask_zdr = zdr_tracks.track_seabreeze.min(dim='timestep', skipna=True) == -1
        seabreeze_mask_kdp = kdp_tracks.track_seabreeze.min(dim='timestep', skipna=True) == -1
        seabreeze_mask_zdr_kdp = zdr_kdp_tracks.track_seabreeze.min(dim='timestep', skipna=True) == -1
        seabreeze_mask_zdr_ltg = zdr_ltg_tracks.track_seabreeze.min(dim='timestep', skipna=True) == -1
        seabreeze_mask_kdp_ltg = kdp_ltg_tracks.track_seabreeze.min(dim='timestep', skipna=True) == -1
        seabreeze_mask_zdr_kdp_ltg = zdr_kdp_ltg_tracks.track_seabreeze.min(dim='timestep', skipna=True) == -1
        seabreeze_mask_ltg = ltg_tracks.track_seabreeze.min(dim='timestep', skipna=True) == -1
    elif seabreeze_side_select2 == 'crossing':
        seabreeze_mask_nothing = (nothing_tracks.track_seabreeze.min(dim='timestep', skipna=True) != -1) & (nothing_tracks.track_seabreeze.max(dim='timestep', skipna=True) != -2)
        seabreeze_mask_zdr = (zdr_tracks.track_seabreeze.min(dim='timestep', skipna=True) != -1) & (zdr_tracks.track_seabreeze.max(dim='timestep', skipna=True) != -2)
        seabreeze_mask_kdp = (kdp_tracks.track_seabreeze.min(dim='timestep', skipna=True) != -1) & (kdp_tracks.track_seabreeze.max(dim='timestep', skipna=True) != -2)
        seabreeze_mask_zdr_kdp = (zdr_kdp_tracks.track_seabreeze.min(dim='timestep', skipna=True) != -1) & (zdr_kdp_tracks.track_seabreeze.max(dim='timestep', skipna=True) != -2)
        seabreeze_mask_zdr_ltg = (zdr_ltg_tracks.track_seabreeze.min(dim='timestep', skipna=True) != -1) & (zdr_ltg_tracks.track_seabreeze.max(dim='timestep', skipna=True) != -2)
        seabreeze_mask_kdp_ltg = (kdp_ltg_tracks.track_seabreeze.min(dim='timestep', skipna=True) != -1) & (kdp_ltg_tracks.track_seabreeze.max(dim='timestep', skipna=True) != -2)
        seabreeze_mask_zdr_kdp_ltg = (zdr_kdp_ltg_tracks.track_seabreeze.min(dim='timestep', skipna=True) != -1) & (zdr_kdp_ltg_tracks.track_seabreeze.max(dim='timestep', skipna=True) != -2)
        seabreeze_mask_ltg = (ltg_tracks.track_seabreeze.min(dim='timestep', skipna=True) != -1) & (ltg_tracks.track_seabreeze.max(dim='timestep', skipna=True) != -2)


    for i, day in enumerate(unique_days):
        nothing_sum[i] = (nothing_tracks.isel(track=seabreeze_mask_nothing).track_day == day).sum()
        zdr_sum[i] = (zdr_tracks.isel(track=seabreeze_mask_zdr).track_day == day).sum()
        kdp_sum[i] = (kdp_tracks.isel(track=seabreeze_mask_kdp).track_day == day).sum()
        zdr_kdp_sum[i] = (zdr_kdp_tracks.isel(track=seabreeze_mask_zdr_kdp).track_day == day).sum()
        zdr_ltg_sum[i] = (zdr_ltg_tracks.isel(track=seabreeze_mask_zdr_ltg).track_day == day).sum()
        kdp_ltg_sum[i] = (kdp_ltg_tracks.isel(track=seabreeze_mask_kdp_ltg).track_day == day).sum()
        zdr_kdp_ltg_sum[i] = (zdr_kdp_ltg_tracks.isel(track=seabreeze_mask_zdr_kdp_ltg).track_day == day).sum()
        ltg_sum[i] = (ltg_tracks.isel(track=seabreeze_mask_ltg).track_day == day).sum()
    unique_days = [d.strftime('%m/%d') for d in unique_days.astype('O')]
    nothing_bar = track_counts_ax.bar(unique_days, nothing_sum, color=styles_dict['nothing']['color'], label=styles_dict['nothing']['label'], linewidth=styles_dict['nothing']['linewidth'])
    rolling_sum += nothing_sum
    zdr_bar = track_counts_ax.bar(unique_days, zdr_sum, bottom=rolling_sum, color=styles_dict['zdr']['color'], label=styles_dict['zdr']['label'], linewidth=styles_dict['zdr']['linewidth'])
    rolling_sum += zdr_sum
    kdp_bar = track_counts_ax.bar(unique_days, kdp_sum, bottom=rolling_sum, color=styles_dict['kdp']['color'], label=styles_dict['kdp']['label'], linewidth=styles_dict['kdp']['linewidth'])
    rolling_sum += kdp_sum
    zdr_kdp_bar = track_counts_ax.bar(unique_days, zdr_kdp_sum, bottom=rolling_sum, color=styles_dict['zdr_kdp']['color'], label=styles_dict['zdr_kdp']['label'], linewidth=styles_dict['zdr_kdp']['linewidth'])
    rolling_sum += zdr_kdp_sum
    zdr_ltg_bar = track_counts_ax.bar(unique_days, zdr_ltg_sum, bottom=rolling_sum, color=styles_dict['zdr_ltg']['color'], label=styles_dict['zdr_ltg']['label'], linewidth=styles_dict['zdr_ltg']['linewidth'])
    rolling_sum += zdr_ltg_sum
    kdp_ltg_bar = track_counts_ax.bar(unique_days, kdp_ltg_sum, bottom=rolling_sum, color=styles_dict['kdp_ltg']['color'], label=styles_dict['kdp_ltg']['label'], linewidth=styles_dict['kdp_ltg']['linewidth'])
    rolling_sum += kdp_ltg_sum
    zdr_kdp_ltg_bar = track_counts_ax.bar(unique_days, zdr_kdp_ltg_sum, bottom=rolling_sum, color=styles_dict['zdr_kdp_ltg']['color'], label=styles_dict['zdr_kdp_ltg']['label'], linewidth=styles_dict['zdr_kdp_ltg']['linewidth'])
    rolling_sum += zdr_kdp_ltg_sum
    ltg_bar = track_counts_ax.bar(unique_days, ltg_sum, bottom=rolling_sum, color=styles_dict['ltg']['color'], label=styles_dict['ltg']['label'], linewidth=styles_dict['ltg']['linewidth'])
    rolling_sum += ltg_sum

    rolling_sum_2 = np.zeros(rolling_sum.shape, dtype=float)
    nothing_frack_bar = track_fracks_ax.bar(unique_days, nothing_sum/rolling_sum, color=styles_dict['nothing']['color'], label=styles_dict['nothing']['label'], linewidth=styles_dict['nothing']['linewidth'])
    rolling_sum_2 += nothing_sum/rolling_sum
    zdr_frack_bar = track_fracks_ax.bar(unique_days, zdr_sum/rolling_sum, bottom=rolling_sum_2, color=styles_dict['zdr']['color'], label=styles_dict['zdr']['label'], linewidth=styles_dict['zdr']['linewidth'])
    rolling_sum_2 += zdr_sum/rolling_sum
    kdp_frack_bar = track_fracks_ax.bar(unique_days, kdp_sum/rolling_sum, bottom=rolling_sum_2, color=styles_dict['kdp']['color'], label=styles_dict['kdp']['label'], linewidth=styles_dict['kdp']['linewidth'])
    rolling_sum_2 += kdp_sum/rolling_sum
    zdr_kdp_frack_bar = track_fracks_ax.bar(unique_days, zdr_kdp_sum/rolling_sum, bottom=rolling_sum_2, color=styles_dict['zdr_kdp']['color'], label=styles_dict['zdr_kdp']['label'], linewidth=styles_dict['zdr_kdp']['linewidth'])
    rolling_sum_2 += zdr_kdp_sum/rolling_sum
    zdr_ltg_frack_bar = track_fracks_ax.bar(unique_days, zdr_ltg_sum/rolling_sum, bottom=rolling_sum_2, color=styles_dict['zdr_ltg']['color'], label=styles_dict['zdr_ltg']['label'], linewidth=styles_dict['zdr_ltg']['linewidth'])
    rolling_sum_2 += zdr_ltg_sum/rolling_sum
    kdp_ltg_frack_bar = track_fracks_ax.bar(unique_days, kdp_ltg_sum/rolling_sum, bottom=rolling_sum_2, color=styles_dict['kdp_ltg']['color'], label=styles_dict['kdp_ltg']['label'], linewidth=styles_dict['kdp_ltg']['linewidth'])
    rolling_sum_2 += kdp_ltg_sum/rolling_sum
    zdr_kdp_ltg_frack_bar = track_fracks_ax.bar(unique_days, zdr_kdp_ltg_sum/rolling_sum, bottom=rolling_sum_2, color=styles_dict['zdr_kdp_ltg']['color'], label=styles_dict['zdr_kdp_ltg']['label'], linewidth=styles_dict['zdr_kdp_ltg']['linewidth'])
    rolling_sum_2 += zdr_kdp_ltg_sum/rolling_sum
    ltg_frack_bar = track_fracks_ax.bar(unique_days, ltg_sum/rolling_sum, bottom=rolling_sum_2, color=styles_dict['ltg']['color'], label=styles_dict['ltg']['label'], linewidth=styles_dict['ltg']['linewidth'])
    if j > 1:
        track_counts_ax.set_xlabel('Day')
        track_fracks_ax.set_xlabel('Day')
    track_counts_ax.xaxis.set_tick_params(rotation=90)
    track_fracks_ax.xaxis.set_tick_params(rotation=90)
    track_fracks_ax.set_title(f'{seabreeze_side_select2} tracks'.title())
    track_counts_ax.set_title(f'{seabreeze_side_select2} tracks'.title())
    track_fracks_ax.set_ylim(0, 1)
    if j == 0:
        axs_counts[-1].legend(handles=[nothing_bar, zdr_bar, kdp_bar, zdr_kdp_bar, zdr_ltg_bar, kdp_ltg_bar, zdr_kdp_ltg_bar, ltg_bar], loc='center', ncols=4)
        axs_counts[-1].axis('off')
        axs_fracks[-1].legend(handles=[nothing_frack_bar, zdr_frack_bar, kdp_frack_bar, zdr_kdp_frack_bar, zdr_ltg_frack_bar, kdp_ltg_frack_bar, zdr_kdp_ltg_frack_bar, ltg_frack_bar], loc='center', ncols=4)
        axs_fracks[-1].axis('off')
    print(f'Sum of Nothing tracks for {seabreeze_side_select2}: {nothing_sum.sum()}')
    print(f'Sum of ZDR tracks for {seabreeze_side_select2}: {zdr_sum.sum()}')
    print(f'Sum of KDP tracks for {seabreeze_side_select2}: {kdp_sum.sum()}')
    print(f'Sum of ZDR/KDP tracks for {seabreeze_side_select2}: {zdr_kdp_sum.sum()}')
    print(f'Sum of ZDR/LTG tracks for {seabreeze_side_select2}: {zdr_ltg_sum.sum()}')
    print(f'Sum of KDP/LTG tracks for {seabreeze_side_select2}: {kdp_ltg_sum.sum()}')
    print(f'Sum of ZDR/KDP/LTG tracks for {seabreeze_side_select2}: {zdr_kdp_ltg_sum.sum()}')
    print(f'Sum of LTG tracks for {seabreeze_side_select2}: {ltg_sum.sum()}')
track_counts_fig.supylabel('Track Count')
track_counts_fig.tight_layout()
# track_counts_fig.suptitle('Category Track Counts by Day')
track_counts_fig.savefig(f'./thesis_figs/all/categorical_hists_stacked/day_counts.pdf')
track_fracks_fig.tight_layout()
# track_fracks_fig.suptitle('Category Track Fractions by Day')
track_fracks_fig.savefig(f'./thesis_figs/all/categorical_hists_stacked_density/day_fracks.pdf')
plt.close(track_counts_fig)
plt.close(track_fracks_fig)

### Bivariate

In [ ]:
bivariate_x_vars = ['track_ccn_profile_0.4', 'track_ccn_profile_0.6', 'track_mlecape', 'track_ll_rh', 'track_min_L2_MCMIPC']
for xdv in bivariate_x_vars:
    ltg_mask = all_tracks.track_flash_count.sum(dim='timestep') > 0
    thing_to_plot_x = all_tracks[xdv].mean(dim='timestep', skipna=True)
    cats_to_plot_x = {
        'nothing' : nothing_tracks[xdv].mean(dim='timestep', skipna=True),
        'zdr' : zdr_tracks[xdv].mean(dim='timestep', skipna=True),
        'kdp' : kdp_tracks[xdv].mean(dim='timestep', skipna=True),
        'zdr_kdp' : zdr_kdp_tracks[xdv].mean(dim='timestep', skipna=True),
        'zdr_ltg' : zdr_ltg_tracks[xdv].mean(dim='timestep', skipna=True),
        'kdp_ltg' : kdp_ltg_tracks[xdv].mean(dim='timestep', skipna=True),
        'zdr_kdp_ltg' : zdr_kdp_ltg_tracks[xdv].mean(dim='timestep', skipna=True),
        'ltg' : ltg_tracks[xdv].mean(dim='timestep', skipna=True)
    }
    if 'isel' in twoDhistDict[xdv]:
        thing_to_plot_x = thing_to_plot_x.isel(**twoDhistDict[xdv]['isel'])
        cats_to_plot_x = {key: val.isel(**twoDhistDict[xdv]['isel']) for key, val in cats_to_plot_x.items()}
    things_to_do_x = twoDhistDict[xdv]
    for ydv, things_to_do_y in twoDhistDict.items():
        for j, this_func in enumerate(things_to_do_y['funcs']):
            cats_to_plot_y = {
                'nothing' : nothing_tracks[ydv],
                'zdr' : zdr_tracks[ydv],
                'kdp' : kdp_tracks[ydv],
                'zdr_kdp' : zdr_kdp_tracks[ydv],
                'zdr_ltg' : zdr_ltg_tracks[ydv],
                'kdp_ltg' : kdp_ltg_tracks[ydv],
                'zdr_kdp_ltg' : zdr_kdp_ltg_tracks[ydv],
                'ltg' : ltg_tracks[ydv]
            }
            thing_to_plot_y = all_tracks[ydv]
            if 'isel' in things_to_do_y:
                for key, val in cats_to_plot_y.items():
                    cats_to_plot_y[key] = val.isel(**things_to_do_y['isel'])
                thing_to_plot_y = thing_to_plot_y.isel(**things_to_do_y['isel'])
            if this_func == 'mean':
                cats_to_plot_y = {key: val.mean(dim='timestep') for key, val in cats_to_plot_y.items()}
                thing_to_plot_y = thing_to_plot_y.mean(dim='timestep')
                fancy_string = ',\nmean along track'
            elif this_func == 'min':
                cats_to_plot_y = {key: val.min(dim='timestep') for key, val in cats_to_plot_y.items()}
                thing_to_plot_y = thing_to_plot_y.min(dim='timestep')
                fancy_string = ',\nmin along track'
            elif this_func == 'max':
                cats_to_plot_y = {key: val.max(dim='timestep') for key, val in cats_to_plot_y.items()}
                thing_to_plot_y = thing_to_plot_y.max(dim='timestep')
                fancy_string = ',\nmax along track'
            elif this_func == 'sum':
                cats_to_plot_y = {key: val.sum(dim='timestep') for key, val in cats_to_plot_y.items()}
                thing_to_plot_y = thing_to_plot_y.sum(dim='timestep')
                fancy_string = ',\nsum along track'
            elif this_func == 'nothing':
                cats_to_plot_y = {key: val for key, val in cats_to_plot_y.items()}
                thing_to_plot_y = thing_to_plot_y
                fancy_string = ''
            else:
                raise ValueError(f"Unknown function {this_func} for variable {ydv}")
            if 'time' in thing_to_plot_y.name:
                thing_to_plot_y = thing_to_plot_y.astype(float)
            if type(things_to_do_x['bins']) == np.ndarray:
                x_bins = things_to_do_x['bins'].copy()
            elif type(things_to_do_x['bins']) == list:
                x_bins = np.array(things_to_do_x['bins'][j]).copy()
            else:
                raise ValueError(f"Unknown bins type {type(things_to_do_x['bins'])} for variable {xdv}")
            x_bin0 = x_bins[0]
            x_binf = x_bins[-1]
            x_bins[0] = np.min([thing_to_plot_x.min(), x_bin0])
            x_bins[-1] = np.max([thing_to_plot_x.max(), x_binf])

            if type(things_to_do_y['bins']) == np.ndarray:
                y_bins = things_to_do_y['bins'].copy()
            elif type(things_to_do_y['bins']) == list:
                y_bins = np.array(things_to_do_y['bins'][j]).copy()
            else:
                raise ValueError(f"Unknown bins type {type(things_to_do_y['bins'])} for variable {ydv}")
            y_bin0 = y_bins[0]
            y_binf = y_bins[-1]
            y_bins[0] = np.min([thing_to_plot_y.min(), y_bin0])
            y_bins[-1] = np.max([thing_to_plot_y.max(), y_binf])

            fig = plt.figure(figsize=(8.5, 8.5))
            gs = GridSpec(3, 2, figure=fig, height_ratios=[1e-4, 1, 1])
            lax = fig.add_subplot(gs[0, :])
            axs = [fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1]), fig.add_subplot(gs[2, 0]), fig.add_subplot(gs[2, 1])]
            count_hist_art = axs[0].hist2d(
                thing_to_plot_x.data,
                thing_to_plot_y.data,
                cmap='viridis', norm=pltcolors.LogNorm(), bins=[x_bins, y_bins], range=[[thing_to_plot_x.min(), thing_to_plot_x.max()], [thing_to_plot_y.min(), thing_to_plot_y.max()]], rasterized=True
            )
            if things_to_do_y.get('scale', None) is not None:
                if 'xlog' in things_to_do_y['scale']:
                    [ax.set_yscale('log') for ax in axs]
                    y_bin0 -= 10**int((np.log10(y_bin0) - 1))
            axs[0].set_xlim(x_bin0, x_binf)
            axs[0].set_ylim(y_bin0, y_binf)
            axs[0].set_title('Distribution of Tracks')
            fig.colorbar(count_hist_art[3], ax=axs[0], label='Track Count', orientation='vertical')
            cat_handles = []
            for i, key in enumerate(cats_to_plot_y.keys()):
                if cats_to_plot_y[key].size == 0:
                    this_cat_handle = axs[1].scatter([np.nan], [np.nan], color=styles_dict[key]['color'], label=styles_dict[key]['label'], s=15, marker='+')
                    cat_handles.append(this_cat_handle)
                    continue
                cat_to_plot_x = cats_to_plot_x[key].data
                cat_to_plot_y = cats_to_plot_y[key].data
                cat_to_plot_x_nonnan = cat_to_plot_x[~np.isnan(cat_to_plot_x) & ~np.isnan(cat_to_plot_y)]
                cat_to_plot_y_nonnan = cat_to_plot_y[~np.isnan(cat_to_plot_x) & ~np.isnan(cat_to_plot_y)]
                if cat_to_plot_x_nonnan.size <= 1:
                    this_cat_handle = axs[1].scatter([np.nan], [np.nan], color=styles_dict[key]['color'], label=styles_dict[key]['label'], s=15, marker='+')
                    cat_handles.append(this_cat_handle)
                    continue
                mu_x, mu_y, ell_x, angle = calculate_error_ellipse(cat_to_plot_x_nonnan, cat_to_plot_y_nonnan)
                this_cat_handle = axs[1].scatter(mu_x, mu_y, color=styles_dict[key]['color'], label=styles_dict[key]['label'], s=15, marker='+')
                cat_handles.append(this_cat_handle)
                ellipse = Ellipse((mu_x, mu_y), width=ell_x[0], height=ell_x[1], angle=angle, color=styles_dict[key]['color'], label=styles_dict[key]['label'], fill=False)
                axs[1].add_patch(ellipse)
            axs[1].set_xlim(x_bin0, x_binf)
            axs[1].set_ylim(y_bin0, y_binf)
            axs[1].set_title('Track Category Error Ellipses')
            lax.legend(handles=cat_handles, ncols=4, loc='center')
            lax.axis('off')
            flash_hist_art = axs[2].hist2d(
                thing_to_plot_x.data,
                thing_to_plot_y.data,
                cmap='viridis', norm=pltcolors.LogNorm(), bins=[x_bins, y_bins], range=[[thing_to_plot_x.min(), thing_to_plot_x.max()], [thing_to_plot_y.min(), thing_to_plot_y.max()]], rasterized=True, weights=all_tracks.track_flash_count.sum(dim='timestep', skipna=True)
            )
            axs[2].set_xlim(x_bin0, x_binf)
            axs[2].set_ylim(y_bin0, y_binf)
            axs[2].set_title('Distribution of Lightning')
            fig.colorbar(flash_hist_art[3], ax=axs[2], label='Track Flash Count', orientation='vertical')

            thing_to_plot_x_ltg = thing_to_plot_x.isel(track=ltg_mask)
            thing_to_plot_y_ltg = thing_to_plot_y.isel(track=ltg_mask)

            n_tracks_ltg = np.histogram2d(thing_to_plot_x_ltg.data, thing_to_plot_y_ltg.data, bins=[x_bins, y_bins], range=[[thing_to_plot_x.min(), thing_to_plot_x.max()], [thing_to_plot_y.min(), thing_to_plot_y.max()]])
            n_tracks_hist = np.histogram2d(thing_to_plot_x.data, thing_to_plot_y.data, bins=[x_bins, y_bins], range=[[thing_to_plot_x.min(), thing_to_plot_x.max()], [thing_to_plot_y.min(), thing_to_plot_y.max()]])
            fraction_ltg = n_tracks_ltg[0] / n_tracks_hist[0]
            ltg_fraction_handle = axs[3].pcolormesh(x_bins, y_bins, fraction_ltg.T, cmap='viridis', rasterized=True)

            fig.colorbar(ltg_fraction_handle, ax=axs[3], label='Fraction of tracks with lightning', orientation='vertical')
            axs[3].set_xlim(x_bin0, x_binf)
            axs[3].set_ylim(y_bin0, y_binf)
            axs[3].set_title('Distribution of Lightning Fraction')
            ylabel_str = f'{all_tracks[ydv].attrs.get("long_name", ydv).replace("brightness", "brightness\n").replace("summed, averaged",
                        "summed,\naveraged").replace("km in fea", "km\n in fea")}{fancy_string} ({all_tracks[ydv].attrs.get("units", "")})'
            fig.supxlabel(f'{all_tracks[xdv].attrs.get("long_name", xdv)}{fancy_string} ({all_tracks[xdv].attrs.get("units", "")})')
            fig.supylabel(ylabel_str)
            fig.tight_layout()
            fig.savefig(f'./thesis_figs/{seabreeze_side_select}/bivariate_ellipse/{xdv}--{ydv}--{this_func}.pdf')
            plt.close(fig)

# Revisions

In [ ]:
date_i_want = dt(2022, 8, 1, 0, 0, 0)
feature_i_want = 18539
pad = 0.01
eightone = xr.open_dataset('/Volumes/LtgSSD/tobac_saves/tobac_Save_20220801/seabreeze-obs.zarr')
feat_time_idx = eightone.feature_time_index.sel(feature=feature_i_want).data
previous_time = eightone.time.isel(time=feat_time_idx - 1).data
eightone = eightone.isel(time=feat_time_idx).sel(feature=feature_i_want)
date_i_want = eightone.time.data.copy().astype('datetime64[us]').astype('O').item()
radar = xr.open_dataset(f'/Volumes/LtgSSD/nexrad_zarr/{date_i_want.strftime('%B').upper()}/{date_i_want.strftime('%Y%m%d')}/KHGX{date_i_want.strftime('%Y%m%d')}_{date_i_want.strftime('%H%M%S')}_V06_grid.zarr').isel(time=0, nradar=0)
lightning = xr.open_dataset(f'/Volumes/LtgSSD/8/6sensor_minimum/LYLOUT_{date_i_want.strftime('%y%m%d')}_000000_86400_map500m.nc')
segmask_bool = (eightone.segmentation_mask == feature_i_want).data
event_mask = ((lightning.event_time.data > previous_time) & (lightning.event_time.data <= eightone.time.data))
flash_mask = ((lightning.flash_time_start.data > previous_time) & (lightning.flash_time_start.data <= eightone.time.data))
mixedphase_mask = (radar.z.data > 4900) & (radar.z.data < 10800)
radar_mixedphase = radar.isel(z=mixedphase_mask)
x2d, y2d = np.meshgrid(eightone.x.data/1000, eightone.y.data/1000)

In [ ]:
high_z_mask = (radar_mixedphase.reflectivity.data > 10)
zdr_col_volume = ((radar_mixedphase.differential_reflectivity > 1) & high_z_mask).sum(dim='z').data.astype(float)
zdr_col_volume[zdr_col_volume == 0] = np.nan
kdp_col_volume = ((radar_mixedphase.KDP_CSU > 0.75) & high_z_mask).sum(dim='z').data.astype(float)
kdp_col_volume[kdp_col_volume == 0] = np.nan

In [ ]:
vmin_ltg = lightning.flash_id[flash_mask].min()
vmax_ltg = lightning.flash_id[flash_mask].max()
fig = plt.figure(figsize=(8, 9.6))
ax1 = fig.add_subplot(2, 2, 1, projection=ccrs.PlateCarree())
ax1.set_extent([eightone.lon.min(), eightone.lon.max(), eightone.lat.min(), eightone.lat.max()], crs=ccrs.PlateCarree())
radar_handle = ax1.pcolormesh(eightone.lon, eightone.lat, radar.reflectivity.max(dim='z'), cmap='ChaseSpectral', vmin=-10, vmax=80, transform=ccrs.PlateCarree(), rasterized=True)
ax1.add_feature(cfeat.STATES, edgecolor='k', facecolor='none')
segmask_bool_nan = segmask_bool.copy().astype(float)
segmask_bool_nan[segmask_bool] = np.nan
feature_rect = [eightone.lon.data[segmask_bool].min() - pad, eightone.lat.data[segmask_bool].min() - pad, eightone.lon.data[segmask_bool].max() + pad, eightone.lat.data[segmask_bool].max() + pad]
ax1.add_patch(Rectangle((feature_rect[0], feature_rect[1]), feature_rect[2] - feature_rect[0], feature_rect[3] - feature_rect[1], edgecolor='red', facecolor='none', lw=2, transform=ccrs.PlateCarree(), label='Feature Bounding Box'))
ax1.scatter(lightning.flash_center_longitude[flash_mask], lightning.flash_center_latitude[flash_mask], c=lightning.flash_id[flash_mask], cmap='tab20', edgecolors='k', s=25, transform=ccrs.PlateCarree(), vmin=vmin_ltg, vmax=vmax_ltg)
ax1.set_title(f'KHGX composite reflectivity,\n feature bounding box, and flash centers\n{date_i_want.strftime("%Y-%m-%d %H:%M:%S")}')
fig.colorbar(radar_handle, ax=ax1, label='Reflectivity (dBZ)', orientation='horizontal')

ax2 = fig.add_subplot(2, 2, 2)
ax2.set_xlim(x2d[segmask_bool].min()-3, x2d[segmask_bool].max()+3)
ax2.set_ylim(y2d[segmask_bool].min()-3, y2d[segmask_bool].max()+3)
ax2.pcolormesh(eightone.x/1000, eightone.y/1000, radar.reflectivity.max(dim='z'), cmap='ChaseSpectral', vmin=-10, vmax=80, rasterized=True)
ax2.pcolormesh(eightone.x/1000, eightone.y/1000, segmask_bool_nan, cmap='Greys', alpha=0.5, rasterized=True)
ltg_handle = ax2.scatter(lightning.event_x[event_mask]/1000, lightning.event_y[event_mask]/1000, c=lightning.event_parent_flash_id[event_mask], cmap='tab20', edgecolors='k', s=25, vmin=vmin_ltg, vmax=vmax_ltg)
ax2.set_title(f'Feature-zoomed composite reflectivity,\nsegmentation mask highlighted,\nLightning VHF events')
fig.colorbar(ltg_handle, ax=ax2, label='Lightning Flash ID', orientation='horizontal')

ax3 = fig.add_subplot(2, 2, 3)
ax3.set_xlim(x2d[segmask_bool].min()-3, x2d[segmask_bool].max()+3)
ax3.set_ylim(y2d[segmask_bool].min()-3, y2d[segmask_bool].max()+3)
zdr_handle = ax3.pcolormesh(eightone.x/1000, eightone.y/1000, zdr_col_volume, cmap='tab20', vmin=0, vmax=9, rasterized=True)
ax3.pcolormesh(x2d, y2d, segmask_bool_nan, cmap='Greys_r', alpha=0.5, rasterized=True)
fig.colorbar(zdr_handle, ax=ax3, label=r'Z$_{DR}$ volume (grid cell count)', orientation='horizontal')
ax3.set_title(r'Z$_{DR}$ column volume,'+'\nfeature segmentation mask')


ax4 = fig.add_subplot(2, 2, 4)
ax4.set_xlim(x2d[segmask_bool].min()-3, x2d[segmask_bool].max()+3)
ax4.set_ylim(y2d[segmask_bool].min()-3, y2d[segmask_bool].max()+3)
kdp_handle = ax4.pcolormesh(eightone.x/1000, eightone.y/1000, kdp_col_volume, cmap='tab20', vmin=0, vmax=9, rasterized=True)
ax4.pcolormesh(x2d, y2d, segmask_bool_nan, cmap='Greys_r', alpha=0.5, rasterized=True)
fig.colorbar(kdp_handle, ax=ax4, label=r'K$_{DP}$ volume (grid cell count)', orientation='horizontal')
ax4.set_title(r'K$_{DP}$ column volume,'+'\nfeature segmentation mask')

[ax.set_xlabel('East-west distance from radar (km)') for ax in [ax2, ax3, ax4]]
[ax.set_ylabel('North-south distance from radar (km)') for ax in [ax2, ax3, ax4]]

fig.suptitle(f'Feature {feature_i_want} | Flash Count: {eightone.feature_flash_count.data.item()} | '+r'Z$_{DR}$' + f' Volume: {eightone.feature_zdrvol.data.item()}' + r' km$^3$ | K$_{DP}$ '+f'Volume: {eightone.feature_kdpvol.data.item()}' + r' km$^3$' + f'\n Max Reflectivity: {eightone.feature_max_reflectivity.data.item():.1f} dBZ | Footprint Area: {eightone.feature_area.data.item()*.25*.25:.3f} km$^2$')
fig.tight_layout()
fig.savefig('./thesis_figs/pol_example.pdf')

In [ ]:
fig, axs = plt.subplots(2, 2)
convt_mean = all_tracks.track_convT.mean(dim='timestep', skipna=True)
convt_mean = convt_mean.copy()
convt_mean[convt_mean == 0] = 1e-11
flash_sum = all_tracks.track_flash_count.sum(dim='timestep', skipna=True)
x_bins = twoDhistDict['track_convT']['bins'].copy()
x_bins = np.append([1e-9], x_bins)
x_bin0 = x_bins[0]
x_binf = x_bins[-1]
x_bins[0] = np.min([thing_to_plot.min(), x_bin0])
x_bins[-1] = np.max([thing_to_plot.max(), x_binf])
y_bins = np.logspace(0, 3, 100)
y_bins = np.append([1e-8], y_bins)
y_bin0 = y_bins[0]
y_binf = y_bins[-1]
y_bins[0] = np.min([flash_sum.min(), y_bin0])
y_bins[-1] = np.max([flash_sum.max(), y_binf])
flash_sum.data[flash_sum.data <= 0.5] = np.nan
axs[0, 0].hist2d(convt_mean, flash_sum, bins=[twoDhistDict['track_convT']['bins'], y_bins], range=[[convt_mean.min(), convt_mean.max()], [flash_sum.min(), flash_sum.max()]], norm=pltcolors.LogNorm())
axs[0, 0].set_ylim(-100, 200)

axs[0, 1].hist2d(convt_mean, flash_sum, bins=[twoDhistDict['track_convT']['bins'], y_bins], range=[[convt_mean.min(), convt_mean.max()], [flash_sum.min(), flash_sum.max()]], norm=pltcolors.LogNorm())
axs[0, 1].set_yscale('log')
axs[0, 1].set_ylim(0.5, 1e3)

In [ ]:
convt_nonnan = convt_mean.data[~np.isnan(convt_mean.data) & ~np.isnan(flash_sum.data)]
flash_sum_nonnan = flash_sum.data[~np.isnan(convt_mean.data) & ~np.isnan(flash_sum.data)]

# reg_base = np.array([x_bin0, x_binf])

this_reg = linregress(convt_nonnan, flash_sum_nonnan)
this_pval_str = f'={this_reg.pvalue:.3f}' if this_reg.pvalue >= 0.001 else f'<.001'
axs[0, 0].plot(reg_base, this_reg.intercept + this_reg.slope*reg_base, color='k', linestyle='--', label=f'y={this_reg.slope:.2f}x+{this_reg.intercept:.1f}\n'+r'r$^{2}$'+f'={this_reg.rvalue**2:.2f}, p{this_pval_str}')
ltg_reg = linregress(convt_nonnan[flash_sum_nonnan >= 1], flash_sum_nonnan[flash_sum_nonnan >= 1])
ltg_pval_str = f'={ltg_reg.pvalue:.3f}' if ltg_reg.pvalue >= 0.001 else f'<.001'
axs[0, 1].plot(reg_base, ltg_reg.intercept + ltg_reg.slope*reg_base, color='r', linestyle='--', label=f'y={ltg_reg.slope:.2f}x+{ltg_reg.intercept:.1f}\n'+r'r$^{2}$'+f'={ltg_reg.rvalue**2:.2f}, p{ltg_pval_str}')

In [ ]:
fig